<a href="https://colab.research.google.com/github/Benevalterjr/ifood/blob/main/ifoodflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 VERSÃO FINAL: Busca Integrada e Endereços do iFood

Este bloco de código é a versão consolidada e funcional. Ele realiza a autenticação, a busca de produtos e a coleta de endereços via GraphQL, exibindo tudo em um único DataFrame organizado.

In [ ]:
# Variáveis de ambiente (substitua pelos valores reais)
os.environ['X_IFOOD_SESSION_ID'] = '05c62e82-a6ff-4ee0-8870-4a5e6a736d3e'
os.environ['X_IFOOD_DEVICE_ID'] = 'a7d4a7c0-bf7c-4fda-b6c4-17843a150064'

In [ ]:
import requests
import json
import os
from pprint import pprint

def get_merchant_details(merchant_id, latitude=-23.5505, longitude=-46.6333):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:145.0) Gecko/20100101 Firefox/145.0",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "pt-BR,pt;q=1",
        "Content-Type": "application/json",
        "Cache-Control": "no-cache, no-store",
        "platform": "Desktop",
        "app_version": "9.129.1",
        "browser": "Windows",
        "X-Ifood-Session-Id": os.getenv('X_IFOOD_SESSION_ID'),
        "X-Ifood-Device-Id": os.getenv('X_IFOOD_DEVICE_ID'),
        "x-device-model": "Windows Firefox",
        "x-client-application-key": "41a266ee-51b7-4c37-9e9d-5cd331f280d5",
        "gps-latitude": str(latitude),
        "gps-longitude": str(longitude),
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-site",
        "Priority": "u=0",
        "Referer": "https://www.ifood.com.br/"
    }

    # GraphQL query extraída do corpo da requisição fornecida
    graphql_query = """
    query ($merchantId: String!) {
        merchant (merchantId: $merchantId, required: true) {
            available
            availableForScheduling
            contextSetup {
                catalogGroup
                context
                regionGroup
            }
            currency
            deliveryFee {
                originalValue
                type
                value
            }
            deliveryMethods {
                catalogGroup
                deliveredBy
                id
                maxTime
                minTime
                mode
                originalValue
                priority
                schedule {
                    now
                    shifts {
                        dayOfWeek
                        endTime
                        interval
                        startTime
                    }
                    timeSlots {
                        availableLoad
                        date
                        endDateTime
                        endTime
                        id
                        isAvailable
                        originalPrice
                        price
                        startDateTime
                        startTime
                    }
                }
                subtitle
                title
                type
                value
                state
            }
            deliveryTime
            distance
            features
            id
            mainCategory {
                code
                name
            }
            minimumOrderValue
            name
            paymentCodes
            preparationTime
            priceRange
            resources {
                fileName
                type
            }
            slug
            tags
            takeoutTime
            userRating
        }
        merchantExtra (merchantId: $merchantId, required: false) {
            address {
                city
                country
                district
                latitude
                longitude
                state
                streetName
                streetNumber
                timezone
                zipCode
            }
            categories {
                code
                description
                friendlyName
            }
            companyCode
            configs {
                bagItemNoteLength
                chargeDifferentToppingsMode
                nationalIdentificationNumberRequired
                orderNoteLength
            }
            deliveryTime
            description
            documents {
                CNPJ {
                    type
                    value
                }
                MCC {
                    type
                    value
                }
            }
            enabled
            features
            groups {
                externalId
                id
                name
                type
            }
            id
            locale
            mainCategory {
                code
                description
                friendlyName
            }
            merchantChain {
                externalId
                id
                name
            }
            metadata {
                ifoodClub {
                    banner {
                        action
                        image
                        priority
                        title
                    }
                }
            }
            minimumOrderValue
            minimumOrderValueV2
            name
            phoneIf
            priceRange
            resources {
                fileName
                type
            }
            shifts {
                dayOfWeek
                duration
                start
            }
            shortId
            tags
            takeoutTime
            test
            type
            userRatingCount
        }
    }
    """

    json_body = {
        "query": graphql_query,
        "variables": {"merchantId": merchant_id}
    }

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "channel": "IFOOD"
    }

    try:
        response = requests.post(
            "https://cw-marketplace.ifood.com.br/v1/merchant-info/graphql",
            headers=headers,
            json=json_body,
            params=params,
            timeout=10
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Erro ao obter detalhes do merchant {merchant_id}: {e}")
        if response is not None:
             try:
                 print("Resposta do servidor (se disponível):", response.json())
             except json.JSONDecodeError:
                 print("Resposta do servidor (texto):", response.text)
        return None

# Exemplo de uso (com um merchantId de exemplo dos resultados anteriores):
# É necessário ter executado a célula anterior para ter 'resultados_busca' disponível
if 'resultados_busca' in locals() and resultados_busca:
    first_merchant_id = None
    # Usa as coordenadas da busca anterior para consistência
    current_latitude = -23.5505
    current_longitude = -46.6333

    try:
        # Tenta extrair o ID do primeiro merchant da estrutura de resultados da busca
        for section in resultados_busca.get('sections', []):
            for card in section.get('cards', []):
                for merchant in card.get('data', {}).get('contents', []):
                    if 'id' in merchant:
                        first_merchant_id = merchant['id']
                        break
                if first_merchant_id:
                    break
            if first_merchant_id:
                break

    except (KeyError, IndexError):
        print("Não foi possível extrair um merchantId dos resultados da busca.")

    if first_merchant_id:
        print(f"Tentando obter detalhes para o merchant ID: {first_merchant_id}")
        merchant_details = get_merchant_details(first_merchant_id, current_latitude, current_longitude)
        if merchant_details:
            print("Detalhes do Merchant (exemplo):")
            # A informação do endereço está aninhada em data['merchantExtra']['address']
            address_info = merchant_details.get('data', {}).get('merchantExtra', {}).get('address')
            if address_info:
                print("Endereço do Estabelecimento:")
                pprint(address_info)
                # Constrói uma string de endereço completa
                full_address = f"{address_info.get('streetName', '')}, {address_info.get('streetNumber', '')} - {address_info.get('district', '')}, {address_info.get('city', '')} - {address_info.get('state', '')}, {address_info.get('zipCode', '')}"
                print(f"Endereço Completo: {full_address}")
            else:
                print("Endereço não encontrado nos detalhes do merchant.")
        else:
            print("Não foi possível obter os detalhes do merchant.")
    else:
        print("Nenhum merchant ID disponível para testar a função get_merchant_details.")
else:
    print("Por favor, execute a busca de produtos ('search_products') primeiro para ter resultados disponíveis.")

Por favor, execute a busca de produtos ('search_products') primeiro para ter resultados disponíveis.


## Código Completo: Busca de Produtos e Detalhes de Endereço (Célula Única)

Este bloco de código integra todas as funcionalidades desenvolvidas para:
1. Configurar as variáveis de ambiente necessárias.
2. Definir a função `search_products` para buscar produtos e estabelecimentos.
3. Definir a função `get_merchant_details` para obter detalhes de endereço de estabelecimentos específicos via GraphQL.
4. Executar a busca inicial de produtos.
5. Extrair IDs únicos de estabelecimentos a partir dos resultados da busca.
6. Buscar os endereços para cada um desses IDs únicos.
7. Combinar os dados dos produtos com os respectivos endereços.
8. Criar e exibir um DataFrame do pandas com todos os dados agregados.

In [ ]:
import requests
import json
from pprint import pprint
import os
import pandas as pd

# --- 1. Configura variáveis de ambiente ---
# Recomenda-se armazenar em secrets ou arquivos .env em produção
%env IFOOD_SESSION_ID = 05c62e82-a6ff-4ee0-8870-4a5e6a736d3e
%env IFOOD_DEVICE_ID = a7d4a7c0-bf7c-4fda-b6c4-17843a150064

os.environ['X_IFOOD_SESSION_ID'] = os.getenv('IFOOD_SESSION_ID')
os.environ['X_IFOOD_DEVICE_ID'] = os.getenv('IFOOD_DEVICE_ID')

# --- 2. Define a função search_products ---
def search_products(latitude=-23.5505, longitude=-46.6333, distance_to=20, term=""):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:130.0) Gecko/20100101 Firefox/130.0',
        'Accept': 'application/json, text/plain, */*',
        'Accept-Language': 'pt-BR,pt;q=1',
        'Content-Type': 'application/json',
        'Cache-Control': 'no-cache, no-store',
        'platform': 'Desktop',
        'app_version': '9.107.24',
        'test_merchants': 'undefined',
        'experiment_details': json.dumps({
            "default_item": {
                "is_discovery_search_service_enabled": True,
                "model_id": "search-optimus-prime-endpoint",
                "engine": "sagemaker",
                "backend_experiment_id": "v4",
                "query_rewriter_rule": "items-synonyms",
                "force_similar_search_disabled": True
            }
        }),
        'experiment_variant': 'default_item',
        'Country': 'BR',
        'browser': 'Windows',
        'X-Ifood-Session-Id': os.getenv('X_IFOOD_SESSION_ID'),
        'X-Ifood-Device-Id': os.getenv('X_IFOOD_DEVICE_ID'),
        'x-device-model': 'Windows Firefox',
        'x-client-application-key': '41a266ee-51b7-4c37-9e9d-5cd331f280d5',
        'gps-latitude': str(latitude),
        'gps-longitude': str(longitude),
        'Sec-Fetch-Dest': 'empty',
        'Sec-Fetch-Mode': 'cors',
        'Sec-Fetch-Site': 'same-site',
        'Priority': 'u=0'
    }

    body = {
        "supported-headers": ["OPERATION_HEADER"],
        "supported-cards": [
            "MERCHANT_LIST", "CATALOG_ITEM_LIST", "CATALOG_ITEM_LIST_V2", "CATALOG_ITEM_LIST_V3",
            "FEATURED_MERCHANT_LIST", "CATALOG_ITEM_CAROUSEL", "CATALOG_ITEM_CAROUSEL_V2",
            "CATALOG_ITEM_CAROUSEL_V3", "BIG_BANNER_CAROUSEL", "IMAGE_BANNER",
            "MERCHANT_LIST_WITH_ITEMS_CAROUSEL", "SMALL_BANNER_CAROUSEL", "NEXT_CONTENT",
            "MERCHANT_CAROUSEL", "MERCHANT_TILE_CAROUSEL", "SIMPLE_MERCHANT_CAROUSEL",
            "INFO_CARD", "MERCHANT_LIST_V2", "ROUND_IMAGE_CAROUSEL", "BANNER_GRID",
            "MEDIUM_IMAGE_BANNER", "MEDIUM_BANNER_CAROUSEL", "RELATED_SEARCH_CAROUSEL",
            "ADS_BANNER"
        ],
        "supported-actions": [
            "catalog-item", "merchant", "page", "card-content", "last-restaurants",
            "webmiddleware", "reorder", "search", "groceries", "home-tab"
        ],
        "feed-feature-name": "",
        "faster-overrides": ""
    }

    params = {
        "alias": "SEARCH_RESULTS_ITEM_TAB_GLOBAL",
        "latitude": latitude,
        "longitude": longitude,
        "channel": "IFOOD",
        "distance_to": distance_to,
        "size": 100,
        "term": term
    }

    try:
        response = requests.post(
            'https://marketplace.ifood.com.br/v2/cardstack/search/results',
            headers=headers,
            json=body,
            params=params,
            timeout=10
        )
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"Erro HTTP: {e}")
        print("Resposta do servidor:")
        print(json.dumps(response.json(), indent=2))
        return None

    return response.json()

# --- 3. Define a função get_merchant_details ---
def get_merchant_details(merchant_id, latitude=-23.5505, longitude=-46.6333):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:145.0) Gecko/20100101 Firefox/145.0",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "pt-BR,pt;q=1",
        "Content-Type": "application/json",
        "Cache-Control": "no-cache, no-store",
        "platform": "Desktop",
        "app_version": "9.129.1",
        "browser": "Windows",
        "X-Ifood-Session-Id": os.getenv('X_IFOOD_SESSION_ID'),
        "X-Ifood-Device-Id": os.getenv('X_IFOOD_DEVICE_ID'),
        "x-device-model": "Windows Firefox",
        "x-client-application-key": "41a266ee-51b7-4c37-9e9d-5cd331f280d5",
        "gps-latitude": str(latitude),
        "gps-longitude": str(longitude),
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-site",
        "Priority": "u=0",
        "Referer": "https://www.ifood.com.br/"
    }

    graphql_query = """
    query ($merchantId: String!) {
        merchant (merchantId: $merchantId, required: true) {
            available
            availableForScheduling
            contextSetup {
                catalogGroup
                context
                regionGroup
            }
            currency
            deliveryFee {
                originalValue
                type
                value
            }
            deliveryMethods {
                catalogGroup
                deliveredBy
                id
                maxTime
                minTime
                mode
                originalValue
                priority
                schedule {
                    now
                    shifts {
                        dayOfWeek
                        endTime
                        interval
                        startTime
                    }
                    timeSlots {
                        availableLoad
                        date
                        endDateTime
                        endTime
                        id
                        isAvailable
                        originalPrice
                        price
                        startDateTime
                        startTime
                    }
                }
                subtitle
                title
                type
                value
                state
            }
            deliveryTime
            distance
            features
            id
            mainCategory {
                code
                name
            }
            minimumOrderValue
            name
            paymentCodes
            preparationTime
            priceRange
            resources {
                fileName
                type
            }
            slug
            tags
            takeoutTime
            userRating
        }
        merchantExtra (merchantId: $merchantId, required: false) {
            address {
                city
                country
                district
                latitude
                longitude
                state
                streetName
                streetNumber
                timezone
                zipCode
            }
            categories {
                code
                description
                friendlyName
            }
            companyCode
            configs {
                bagItemNoteLength
                chargeDifferentToppingsMode
                nationalIdentificationNumberRequired
                orderNoteLength
            }
            deliveryTime
            description
            documents {
                CNPJ {
                    type
                    value
                }
                MCC {
                    type
                    value
                }
            }
            enabled
            features
            groups {
                externalId
                id
                name
                type
            }
            id
            locale
            mainCategory {
                code
                description
                friendlyName
            }
            merchantChain {
                externalId
                id
                name
            }
            metadata {
                ifoodClub {
                    banner {
                        action
                        image
                        priority
                        title
                    }
                }
            }
            minimumOrderValue
            minimumOrderValueV2
            name
            phoneIf
            priceRange
            resources {
                fileName
                type
            }
            shifts {
                dayOfWeek
                duration
                start
            }
            shortId
            tags
            takeoutTime
            test
            type
            userRatingCount
        }
    }
    """

    json_body = {
        "query": graphql_query,
        "variables": {"merchantId": merchant_id}
    }

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "channel": "IFOOD"
    }

    try:
        response = requests.post(
            "https://cw-marketplace.ifood.com.br/v1/merchant-info/graphql",
            headers=headers,
            json=json_body,
            params=params,
            timeout=10
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Erro ao obter detalhes do merchant {merchant_id}: {e}")
        if response is not None:
             try:
                 print("Resposta do servidor (se disponível):", response.json())
             except json.JSONDecodeError:
                 print("Resposta do servidor (texto):", response.text)
        return None

# --- 4. Lógica Principal: Busca de produtos, coleta de endereços e combinação de dados ---
current_latitude = -23.5505
current_longitude = -46.6333
term_search = "vinho branco"

# Executa a busca de produtos
resultados_busca = search_products(
    latitude=current_latitude,
    longitude=current_longitude,
    term=term_search
)

if not resultados_busca:
    print("A busca inicial não retornou resultados ou ocorreu um erro.")
else:
    print(f"Busca por '{term_search}' retornou resultados.")

    # Coleta IDs de estabelecimentos únicos
    unique_merchant_ids = set()
    if 'sections' in resultados_busca:
        for section in resultados_busca['sections']:
            if 'cards' in section:
                for card in section['cards']:
                    if 'data' in card and 'contents' in card['data']:
                        for merchant in card['data']['contents']:
                            if 'id' in merchant:
                                unique_merchant_ids.add(merchant['id'])

    print(f"\nTotal de IDs de estabelecimentos únicos encontrados: {len(unique_merchant_ids)}")

    # Busca os endereços para cada estabelecimento único
    merchant_addresses = {}
    for merchant_id in unique_merchant_ids:
        details = get_merchant_details(merchant_id, current_latitude, current_longitude)
        if details and details.get('data', {}).get('merchantExtra', {}).get('address'):
            address_info = details['data']['merchantExtra']['address']
            merchant_addresses[merchant_id] = address_info

    print(f"\nTotal de endereços de estabelecimentos obtidos: {len(merchant_addresses)}")

    # Combina os dados de produtos com os endereços
    combined_data = []
    if 'sections' in resultados_busca:
        for section in resultados_busca['sections']:
            if 'cards' in section:
                for card in section['cards']:
                    if 'data' in card and 'contents' in card['data']:
                        for merchant in card['data']['contents']:
                            merchant_id = merchant.get('id')
                            merchant_address_info = merchant_addresses.get(merchant_id, {})

                            for item in merchant.get('items', []):
                                product_data = {
                                    'merchantName': merchant.get('name'),
                                    'merchantId': merchant_id,
                                    'deliveryFee': merchant.get('deliveryInfo', {}).get('fee', 'N/A'),
                                    'distance': merchant.get('distance', 'N/A'),
                                    'rating': merchant.get('userRating', 0),
                                    'productName': item.get('name'),
                                    'productId': item.get('id'),
                                    'price': item.get('pricing', {}).get('unitPrice')
                                }
                                # Adiciona os detalhes do endereço
                                product_data['streetName'] = merchant_address_info.get('streetName', 'N/A')
                                product_data['streetNumber'] = merchant_address_info.get('streetNumber', 'N/A')
                                product_data['district'] = merchant_address_info.get('district', 'N/A')
                                product_data['city'] = merchant_address_info.get('city', 'N/A')
                                product_data['state'] = merchant_address_info.get('state', 'N/A')
                                product_data['zipCode'] = merchant_address_info.get('zipCode', 'N/A')
                                product_data['address_latitude'] = merchant_address_info.get('latitude', 'N/A')
                                product_data['address_longitude'] = merchant_address_info.get('longitude', 'N/A')

                                combined_data.append(product_data)

    print(f"\nTotal de entradas de produtos combinadas: {len(combined_data)}")

    # Cria e exibe o DataFrame
    df_combined = pd.DataFrame(combined_data)
    display(df_combined)

env: IFOOD_SESSION_ID=05c62e82-a6ff-4ee0-8870-4a5e6a736d3e
env: IFOOD_DEVICE_ID=a7d4a7c0-bf7c-4fda-b6c4-17843a150064
Busca por 'vinho branco' retornou resultados.

Total de IDs de estabelecimentos únicos encontrados: 100

Total de endereços de estabelecimentos obtidos: 100

Total de entradas de produtos combinadas: 396


,merchantName,merchantId,deliveryFee,distance,rating,productName,productId,price,streetName,streetNumber,district,city,state,zipCode,address_latitude,address_longitude
0,Daki Mercado Express | Liberdade,e2f158f2-b04c-427e-a0ce-d672c98b2522,1499,1.29,4.9,Vinho Branco Chileno Sauvignon Blanc Três Meda...,0899b614-2e15-4980-8dbc-f494bb765473,4399,Rua Galvão Bueno,714,Liberdade,SAO PAULO,SP,01506000,-23.562017,-46.634993
1,Daki Mercado Express | Liberdade,e2f158f2-b04c-427e-a0ce-d672c98b2522,1499,1.29,4.9,Vinho Branco Chileno Reservado Sauvignon Conch...,1aa1151e-11a3-4407-941e-8ebdece88a33,3579,Rua Galvão Bueno,714,Liberdade,SAO PAULO,SP,01506000,-23.562017,-46.634993
2,Daki Mercado Express | Liberdade,e2f158f2-b04c-427e-a0ce-d672c98b2522,1499,1.29,4.9,Vinho Chileno Chilano Chardonnay Branco Seco 7...,73a95689-254b-44d4-9a86-0c49afbe885b,3999,Rua Galvão Bueno,714,Liberdade,SAO PAULO,SP,01506000,-23.562017,-46.634993
3,Daki Mercado Express | Liberdade,e2f158f2-b04c-427e-a0ce-d672c98b2522,1499,1.29,4.9,Vinho Branco Quinta de Bons Ventos Garrafa 750ml,0464ee5a-511c-4e0f-8113-17602c5d0494,6479,Rua Galvão Bueno,714,Liberdade,SAO PAULO,SP,01506000,-23.562017,-46.634993
4,Daki Mercado Express | Liberdade,e2f158f2-b04c-427e-a0ce-d672c98b2522,1499,1.29,4.9,Vinho Branco Chileno 120 Santa Rita Chardonnay...,0af273dc-436c-4aa4-a9e1-4a5878058419,5299,Rua Galvão Bueno,714,Liberdade,SAO PAULO,SP,01506000,-23.562017,-46.634993
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391,Boteco Santa Cruz 01,1815cf1a-5e53-4699-ab9f-212c310ab08f,899,5.13,4.8,Vinho Dom Bosco Branco Suave 750ml *,917a5482-8d91-4353-be8a-bedd4454fa3d,2675,Rua Domingos de Morais,2330,Vila Mariana,SAO PAULO,SP,04036000,-23.596693,-46.636888
392,Boteco Santa Cruz 01,1815cf1a-5e53-4699-ab9f-212c310ab08f,899,5.13,4.8,Vinho Quinta Do Morgado Branco Seco 750ml *,fe96c4bf-4172-4b87-ac22-a0fbc8101897,3000,Rua Domingos de Morais,2330,Vila Mariana,SAO PAULO,SP,04036000,-23.596693,-46.636888
393,Boteco Santa Cruz 01,1815cf1a-5e53-4699-ab9f-212c310ab08f,899,5.13,4.8,Vinho Periquita Branco 750ml *,9af75c25-8fea-4e8d-ad25-96b16312d0a8,9300,Rua Domingos de Morais,2330,Vila Mariana,SAO PAULO,SP,04036000,-23.596693,-46.636888
394,Boteco Santa Cruz 01,1815cf1a-5e53-4699-ab9f-212c310ab08f,899,5.13,4.8,Vinho Chilano Sauvignon Blanc 750ml *,ef4f5d8d-9b27-401e-a141-a37aeddda814,5314,Rua Domingos de Morais,2330,Vila Mariana,SAO PAULO,SP,04036000,-23.596693,-46.636888


In [ ]:
import requests
import json
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import time

class IFoodOptimizer:
    def __init__(self):
        # IDs que você confirmou que funcionam
        self.session_id = "05c62e82-a6ff-4ee0-8870-4a5e6a736d3e"
        self.device_id = "a7d4a7c0-bf7c-4fda-b6c4-17843a150064"
        self.app_key = "41a266ee-51b7-4c37-9e9d-5cd331f280d5"

        # Criamos uma sessão persistente para ganhar velocidade
        self.session = requests.Session()
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:130.0) Gecko/20100101 Firefox/130.0',
            'Accept': 'application/json, text/plain, */*',
            'Content-Type': 'application/json',
            'platform': 'Desktop',
            'app_version': '9.140.2',
            'X-Ifood-Session-Id': self.session_id,
            'X-Ifood-Device-Id': self.device_id,
            'x-client-application-key': self.app_key,
        }
        self.session.headers.update(self.headers)

    def search_products(self, term, lat, lon):
        url = 'https://marketplace.ifood.com.br/v2/cardstack/search/results'
        params = {
            "alias": "SEARCH_RESULTS_ITEM_TAB_GLOBAL",
            "latitude": lat, "longitude": lon,
            "channel": "IFOOD", "size": 50, "term": term
        }
        body = {
            "supported-headers": ["OPERATION_HEADER"],
            "supported-cards": ["MERCHANT_LIST_WITH_ITEMS_CAROUSEL", "CATALOG_ITEM_LIST_V2"],
            "supported-actions": ["catalog-item", "merchant", "search"]
        }

        print(f"🔎 Buscando: {term}...")
        response = self.session.post(url, params=params, json=body)
        return response.json()

    def get_merchant_details(self, merchant_id, lat, lon):
        """Busca detalhes de um único merchant (usado em paralelo)"""
        url = "https://cw-marketplace.ifood.com.br/v1/merchant-info/graphql"
        query = """
        query ($id: String!) {
            merchantExtra (merchantId: $id) {
                address { streetName streetNumber district city zipCode latitude longitude }
                documents { CNPJ { value } }
            }
        }
        """
        payload = {"query": query, "variables": {"id": merchant_id}}
        params = {"latitude": lat, "longitude": lon, "channel": "IFOOD"}

        try:
            res = self.session.post(url, json=payload, params=params, timeout=5)
            data = res.json().get('data', {}).get('merchantExtra', {})
            return merchant_id, data.get('address', {}), data.get('documents', {}).get('CNPJ', {}).get('value')
        except:
            return merchant_id, {}, None

    def run(self, term, lat, lon):
        start_time = time.time()

        # 1. Busca inicial
        search_data = self.search_products(term, lat, lon)

        # 2. Extração de IDs Únicos
        merchant_ids = set()
        product_list = []

        for section in search_data.get('sections', []):
            for card in section.get('cards', []):
                contents = card.get('data', {}).get('contents', [])
                for merchant in contents:
                    m_id = merchant.get('id')
                    if m_id:
                        merchant_ids.add(m_id)
                        # Salva dados parciais do produto
                        for item in merchant.get('items', []):
                            product_list.append({
                                'merchantId': m_id,
                                'merchantName': merchant.get('name'),
                                'distance': merchant.get('distance'),
                                'rating': merchant.get('userRating'),
                                'productName': item.get('name'),
                                'price': item.get('pricing', {}).get('unitPrice', 0) / 100
                            })

        print(f"📦 Encontrados {len(product_list)} produtos em {len(merchant_ids)} restaurantes.")

        # 3. Busca de endereços em PARALELO (Otimização de Velocidade)
        print(f"🚀 Coletando endereços em paralelo...")
        address_map = {}
        cnpj_map = {}

        with ThreadPoolExecutor(max_workers=10) as executor:
            # Dispara as requisições simultaneamente
            futures = [executor.submit(self.get_merchant_details, mid, lat, lon) for mid in merchant_ids]
            for future in futures:
                mid, addr, cnpj = future.result()
                address_map[mid] = addr
                cnpj_map[mid] = cnpj

        # 4. Combinação Final
        for prod in product_list:
            addr = address_map.get(prod['merchantId'], {})
            prod.update({
                'Rua': addr.get('streetName', 'N/A'),
                'Num': addr.get('streetNumber', 'N/A'),
                'Bairro': addr.get('district', 'N/A'),
                'Cidade': addr.get('city', 'N/A'),
                'CEP': addr.get('zipCode', 'N/A'),
                'CNPJ': cnpj_map.get(prod['merchantId'], 'N/A'),
                'Lat_Restaurante': addr.get('latitude', 'N/A'),
                'Lon_Restaurante': addr.get('longitude', 'N/A')
            })

        end_time = time.time()
        print(f"✨ Concluído em {end_time - start_time:.2f} segundos.")
        return pd.DataFrame(product_list)

# --- EXECUÇÃO ---
optimizer = IFoodOptimizer()
df_final = optimizer.run(term="vinho tinto", lat=-23.5960, lon=-46.6467)

# Exibe os resultados mais baratos
display(df_final.sort_values(by='price').head(2))

🔎 Buscando: vinho tinto...
📦 Encontrados 496 produtos em 100 restaurantes.
🚀 Coletando endereços em paralelo...
✨ Concluído em 2.36 segundos.


,merchantId,merchantName,distance,rating,productName,price,Rua,Num,Bairro,Cidade,CEP,CNPJ,Lat_Restaurante,Lon_Restaurante
320,8437323c-0164-4586-9e46-a147281966cb,Conveniencia Menezes,8.29,4.9,Vinagre de Vinho Tinto Neval 750ml,11.20,Rua Guiomar Branco da Silva,548,Vila Marari,SAO PAULO,04402190,45726950000136,-23.669328,-46.661232
495,a3cf8a1c-08e9-48e3-9724-5663e39a2444,Mercado Remax,4.89,4.7,Vinho Tinto Brasileiro Chalise Suave Garrafa 7...,17.53,R BELA CINTRA,413,CERQUEIRA CESAR,SAO PAULO,01415000,00013026000174,-23.552999,-46.656534


In [ ]:
import folium
from folium.plugins import MarkerCluster

# Localização do usuário (baseada na busca anterior)
user_lat, user_lon = -23.5960, -46.6467

# Filtrar apenas estabelecimentos que possuem coordenadas válidas
df_mapa = df_final[df_final['Lat_Restaurante'] != 'N/A'].copy()

# Criar o mapa centralizado na média das coordenadas encontradas
map_center = [df_mapa['Lat_Restaurante'].mean(), df_mapa['Lon_Restaurante'].mean()]

# Inicializa o mapa sem os tiles padrão
m = folium.Map(location=map_center, zoom_start=13, tiles=None)

# Adiciona o TileLayer CartoDB Positron (light_all)
folium.TileLayer(
    tiles='https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>',
    subdomains='abcd',
    max_zoom=20,
    name='CartoDB.Positron'
).add_to(m)

# Adiciona o Pinpoint da localização do usuário (Casa)
folium.Marker(
    location=[user_lat, user_lon],
    popup='Minha Localização',
    tooltip='Sua Casa',
    icon=folium.Icon(color='blue', icon='home', prefix='fa')
).add_to(m)

# Usar cluster para não poluir o mapa para as lojas
marker_cluster = MarkerCluster().add_to(m)

for idx, row in df_mapa.iterrows():
    popup_text = f"""
    <b>Loja:</b> {row['merchantName']}<br>
    <b>Produto:</b> {row['productName']}<br>
    <b>Preço:</b> R$ {row['price']:.2f}<br>
    <b>Endereço:</b> {row['Rua']}, {row['Num']}
    """
    folium.Marker(
        location=[row['Lat_Restaurante'], row['Lon_Restaurante']],
        popup=folium.Popup(popup_text, max_width=300),
        tooltip=row['merchantName'],
        icon=folium.Icon(color='red', icon='shopping-cart', prefix='fa')
    ).add_to(marker_cluster)

# Exibir o mapa
m